|<h2>Substack post:</h2>|<h1><a href=" " target="_blank">Explore LLM word representations using similarity analysis (part 1)</a></h1>|
|-|:-:|
|<h2>Author:<h2>|<h1>Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h1>|

<br>

<i>Using the code without reading the post may lead to confusion or errors.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

In [ ]:
### matplotlib adjustments (commented lines are for dark mode)

# svg plots (higher-res)
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

plt.rcParams.update({
    'figure.facecolor': '#282a2c',
    'figure.edgecolor': '#282a2c',
    'axes.facecolor':   '#282a2c',
    'axes.edgecolor':   '#DDE2F4',
    'axes.labelcolor':  '#DDE2F4',
    'xtick.color':      '#DDE2F4',
    'ytick.color':      '#DDE2F4',
    'text.color':       '#DDE2F4',
    'axes.spines.right': False,
    'axes.spines.top':   False,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold',
    'savefig.dpi':300,
})

# **Extract embeddings from two LLMs**

In [ ]:
# BERT tokenizer and model
from transformers import BertTokenizer, BertModel

# load BERT tokenizer and model
tokenizerB = BertTokenizer.from_pretrained('bert-large-uncased')
modelB = BertModel.from_pretrained('bert-large-uncased')

# the embeddings matrix
embeddingsB = modelB.embeddings.word_embeddings.weight.detach()

In [ ]:
# GPT2 tokenizer and model
from transformers import AutoTokenizer,GPT2Model
tokenizerG = AutoTokenizer.from_pretrained('gpt2-medium')
modelG = GPT2Model.from_pretrained('gpt2-medium')

# the embeddings matrix
embeddingsG = modelG.wte.weight.detach()

In [ ]:
print(f'BERT vocabulary size: {tokenizerB.vocab_size}')
print(f'GPT2 vocabulary size: {tokenizerG.vocab_size}')
print('')

print(f'BERT embeddings shape: {list(embeddingsB.shape)}')
print(f'GPT2 embeddings shape: {list(embeddingsG.shape)}')

# **Directly comparing model embeddings**

In [ ]:
# list of words for RSA
words = [ 'galaxy', 'asteroid', 'comet', 'cosmos', 'space', 'sun', 'planet', 'moon', 'star', 'orbit',
          'ceiling', 'sofa', 'couch', 'carpet', 'door', 'window', 'lamp', 'chair', 'table', 'rug', 'bed', 'floor', 'wall',
          'pear', 'grape', 'banana', 'cherry', 'peach', 'apple', 'seed', 'jelly', 'orange', 'lime', 'fruit'
        ]


# will be convenient later
num_words = len(words)

# initialize lists of tokens
tokensB = []
tokensG = []


print('    Word   |  BERT | GTP-2')
print('-----------+-------+-------')

for w in words:

  # tokenize this word
  tok_b = tokenizerB.encode(w,add_special_tokens=False)[0]
  tok_g = tokenizerG.encode(f' {w}')[0]

  # print
  print(f'{w:>10} | {tok_b:5} | {tok_g:5}')

  # add to list
  tokensB.append(tok_b)
  tokensG.append(tok_g)

In [ ]:
# embeddings matrix for these words
sub_embedB = embeddingsB[tokensB,:]
sub_embedG = embeddingsG[tokensG,:]

print(f'BERT embeddings shape: {list(sub_embedB.shape)}')
print(f'GPT2 embeddings shape: {list(sub_embedG.shape)}')

In [ ]:
fig = plt.figure(figsize=(12,4))
gs = gridspec.GridSpec(1,3,figure=fig)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1:])


# scatter plot of an example word
ax1.plot(sub_embedB[0,:],sub_embedG[0,:],'kh',
            markerfacecolor=[.7,.7,.9,.7])
ax1.set(xlabel='BERT embeddings',ylabel='GPT-2 embeddings',title=f'Embeddings for "{words[0]}"')

# correlations for all words
for i in range(num_words):
  r = np.corrcoef(sub_embedB[i,:],sub_embedG[i,:])[0,1]
  ax2.plot(i,r,'kh',markerfacecolor=[.9,.7,.7],markersize=12)

ax2.axhline(0,linestyle='--',color='w',linewidth=.5,zorder=-1)
ax2.set(xticks=range(num_words),xticklabels=words,ylim=[-.5,.5],
           title='Correlation for each word',ylabel='Correlation coefficient')
ax2.tick_params(axis='x',labelrotation=90)

plt.tight_layout()
plt.savefig('RSA_LLMs_part1_fig2.svg')
plt.show()

# **Representational similarity analysis**

In [ ]:
# cosine similarities
csG = torch.zeros((num_words,num_words))
csB = torch.zeros((num_words,num_words))

for i in range(num_words):
  csB[i,:] = torch.cosine_similarity(sub_embedB[i],sub_embedB)
  csG[i,:] = torch.cosine_similarity(sub_embedG[i],sub_embedG)


# extract the upper-triangular elements
unique_B = csB[np.triu_indices_from(csB, k=1)]
unique_G = csG[np.triu_indices_from(csG, k=1)]

# Pearson correlation
r = np.corrcoef(unique_B,unique_G)[0,1]

print(f'Size of similarity matrices: {csB.shape}')
print(f'Number of non-redundant elements: {len(unique_B)}')

In [ ]:
# and visualize
fig,axs = plt.subplots(1,2,figsize=(10,4))

# color limits for images
vminmax = [.1,.5]

# BERT: cosine similarity matrix
h = axs[0].imshow(csB,vmin=vminmax[0],vmax=vminmax[1],cmap='plasma')
axs[0].set(xticks=range(0,len(words),2),xticklabels=words[::2],
           yticks=range(1,len(words),2),yticklabels=words[1::2],
           title='Cossim matrix for BERT')
axs[0].tick_params(axis='x',labelrotation=90)
fig.colorbar(h,ax=axs[0],fraction=.046,pad=.02)


# GPT2: cosine similarity matrix
h = axs[1].imshow(csG,vmin=vminmax[0],vmax=vminmax[1],cmap='plasma')
axs[1].set(xticks=range(0,len(words),2),xticklabels=words[::2],
           yticks=range(1,len(words),2),yticklabels=words[1::2],
           title='Cossim matrix for GPT-2')
axs[1].tick_params(axis='x',labelrotation=90)
fig.colorbar(h,ax=axs[1],fraction=.046,pad=.02)

plt.tight_layout()
plt.savefig('RSA_LLMs_part1_fig3.svg')
plt.show()

In [ ]:
# scatter plot
_,ax = plt.subplots(1,figsize=(6,5))

ax.plot(unique_B,unique_G,'wh',markersize=8,markerfacecolor=[.7,.9,.7,.5])
ax.set(xlabel='BERT cosine similarities',ylabel='GPT-2 cosine similarities',
              title=f'Correlation (RSA score): r = {r:.2f}')
ax.grid(linestyle='--',color=[.3,.3,.3],linewidth=.4,zorder=-10)

plt.tight_layout()
plt.savefig('RSA_LLMs_part1_fig4.svg')
plt.show()

In [ ]:
## simple analogy

sevenA = np.array([
    [0,0,0,0,0,0,0,0],
    [1,1,1,1,1,1,1,0],
    [0,0,0,0,0,1,0,0],
    [0,0,0,0,1,0,0,0],
    [0,0,0,1,0,0,0,0],
    [0,0,1,0,0,0,0,0],
    [0,1,0,0,0,0,0,0],
    [1,0,0,0,0,0,0,0]
],dtype=float)

# rotated copy
sevenB = sevenA.T

# add some noise
sevenA += np.random.rand(*sevenA.shape)*.4
sevenB += np.random.rand(*sevenB.shape)*.4


_,axs = plt.subplots(1,3,figsize=(10,3.5))
axs[0].imshow(sevenA,cmap='magma')
axs[0].set(title='Embeddings "A"')

axs[1].imshow(sevenB,cmap='magma')
axs[1].set(title='Embeddings "B"')

axs[2].plot(sevenA,sevenB,'wh',markerfacecolor=[.7,.9,.9,.7])
axs[2].set(xlabel='Embeddings "A"',ylabel='Embeddings "B"',title='Direct comparison')

plt.tight_layout()
plt.savefig('RSA_LLMs_part1_fig5.svg')
plt.show()

# **RSA with different embeddings sizes**